# SMA thermal session explorer

Interactive Plotly rebuild of the plotting in `analyze_sma.py` + `lib_analysis.py`.
Zoom/pan with the mouse; **save any view** with the camera icon in the figure's
top-right toolbar (exports the *current* zoomed view as PNG).

Three parts, each an independent cell you can edit / add / drop panels in:

1. **Raw sanity** — no conversion, in mV / mA. Just "are the readings sane?"
   `V_LDO`, `I_SMA`, `V_laser`, `V_load`. ADC-rail saturation flagged red.
2. **Converted + actuation-marked** — R [Ω], P [W], displacement [mm], force [mN],
   and ΔR/R₀ [%]. Fire windows shaded.
3. **Cross-plots** — displacement & force vs resistance (shared x), and vs power
   (shared x). Colored by fire-vs-cool phase to reveal the hysteresis loop.

All loaders, calibration, clock-alignment and segmentation are imported from
`lib_analysis.py` — single-sourced, so this notebook can't drift from the scripts.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Reuse the single-sourced loaders/segmentation/clock-alignment from lib_analysis.py.
# (Importing it pulls in matplotlib on the Agg backend — harmless; we use Plotly.)
HERE = Path.cwd()
if not (HERE / "lib_analysis.py").exists():
    raise RuntimeError(f"open this notebook from the module dir (has lib_analysis.py); cwd={HERE}")
sys.path.insert(0, str(HERE))
import lib_analysis as sp

# Plotly toolbar: high-DPI PNG export via the camera icon.
def png_cfg(name):
    return {"toImageButtonOptions": {"format": "png", "scale": 3, "filename": name},
            "displaylogo": False}

In [ ]:
# ── choose the session ───────────────────────────────────────────────────
# None -> newest finalized session under data/;  or set e.g.
# SESSION = "data/console_20260715_193936_5V0.5V"
SESSION = None

sess = Path(SESSION) if SESSION else sp.latest_session()
if sess is None:
    raise RuntimeError("no session found under data/console_*")
print("session:", sess.name)

In [ ]:
# ── load, calibrate, align clocks, segment ───────────────────────────────
h7 = sp.load_h7(sess / "h7.csv")
meta, k, v0, lscale, loff, cold_r = sp.load_meta(sess)
cmd = sp.load_cycle_cmd(sess / "events.csv")
off_us = sp.load_m4_to_m7_offset_us(sess)   # M4 (laser/load) -> M7 (SMA) timeline

# firmware-clock time bases (seconds); laser/load shifted onto the SMA timeline
t_v   = sp.timebase(h7["sma_v"])
t_i   = sp.timebase(h7["sma_i"])
t_r   = sp.timebase(h7["sma_r"])
t_las = sp.timebase(h7["laser"], off_us)
t_lod = sp.timebase(h7["load"],  off_us)
t0 = t_v[0]                                  # session reference (M7)
def rel(tb):
    return tb - t0

# raw channels (SI as streamed)
V_ldo = h7["sma_v"]["v"]     # V   (LDO output across the SMA)
I_sma = h7["sma_i"]["v"]     # A
V_las = h7["laser"]["v"]     # V   (raw ADC)
V_lod = h7["load"]["v"]      # V   (raw ADC)
R_ohm = h7["sma_r"]["v"]     # Ω   (streamed directly)

# conversions (constants from config.yaml via lib_analysis.load_meta)
disp_mm  = ((V_las * 1000.0 - v0) / k) / 1000.0      # mm
force_mN = lscale * (V_lod - loff) * 1000.0          # mN

# actuation segmentation
if cmd is None:
    print("WARNING: no `cycle ...` command in events.csv — fire windows disabled")
    onsets = np.array([]); fire_s = 0.0
else:
    onsets = sp.find_fire_onsets(t_v, V_ldo, cmd.v_high, cmd.v_low)
    fire_s = cmd.fire_s
    print(f"actuation: {cmd.n} cycles, fire {cmd.fire_ms:.0f} ms @ {cmd.v_high} V, "
          f"cool {cmd.cool_ms:.0f} ms @ {cmd.v_low} V — found {len(onsets)} onset(s)")
onsets_rel = onsets - t0

# cold resistance reference R0 for ΔR/R₀
if cold_r:
    R0 = float(cold_r)
elif onsets.size:
    pre = rel(t_r) < (onsets_rel[0] - 0.05)
    R0 = float(np.median(R_ohm[pre])) if pre.any() else float(np.median(R_ohm))
else:
    R0 = float(np.median(R_ohm))
print(f"R0 = {R0:.4f} Ω   |   M4->M7 offset = {off_us/1e6:.3f} s")

In [ ]:
# ── co-sampled grid for the Part-3 cross-plots ───────────────────────────
# R, P, displacement, force are on different channels/clocks and sample rates;
# to plot one against another they must share sample instants. Interpolate all
# onto one uniform time grid over the overlapping span.
FS_GRID = 200.0                              # Hz; ~ the effective ADC rate
g0 = max(rel(t_v)[0], rel(t_r)[0], rel(t_las)[0], rel(t_lod)[0], 0.0)
g1 = min(rel(t_v)[-1], rel(t_r)[-1], rel(t_las)[-1], rel(t_lod)[-1])
grid = np.arange(g0, g1, 1.0 / FS_GRID)

Rg     = np.interp(grid, rel(t_r),   R_ohm)
Vg     = np.interp(grid, rel(t_v),   V_ldo)
Ig     = np.interp(grid, rel(t_i),   I_sma)
Pg     = Vg * Ig                              # W
dispg  = np.interp(grid, rel(t_las), disp_mm)
forceg = np.interp(grid, rel(t_lod), force_mN)

# fire-vs-cool phase on the grid
fire_mask = np.zeros(grid.shape, dtype=bool)
for ot in onsets_rel:
    fire_mask |= (grid >= ot) & (grid < ot + fire_s)
print(f"grid: {grid.size} pts over {g0:.2f}..{g1:.2f} s  ({fire_mask.mean()*100:.0f}% in-fire)")

## Part 1 — Raw sanity check (no conversion)

Straight from the ADC, in mV / mA. Red dots = samples pinned to the ±5 V rail
(a saturated raw reading — the thing this view exists to catch).

In [ ]:
RAIL_mV = 4990.0   # |reading| at/above this ~ pinned to the ±5 V ADC rail

def _sat(x, y_mV):
    m = np.abs(y_mV) >= RAIL_mV
    return x[m], y_mV[m]

rows = [
    ("V_LDO  (SMA drive) [mV]", rel(t_v),   V_ldo * 1e3, sp.C_VOLT,  True),
    ("I_SMA [mA]",              rel(t_i),   I_sma * 1e3, sp.C_CURR,  False),
    ("V_laser  (raw ADC) [mV]", rel(t_las), V_las * 1e3, sp.C_DISP,  True),
    ("V_load  (raw ADC) [mV]",  rel(t_lod), V_lod * 1e3, sp.C_FORCE, True),
]
fig = make_subplots(rows=len(rows), cols=1, shared_xaxes=True, vertical_spacing=0.035,
                    subplot_titles=[r[0] for r in rows])
for i, (title, x, y, color, flag_sat) in enumerate(rows, start=1):
    fig.add_trace(go.Scattergl(x=x, y=y, mode="lines", line=dict(color=color, width=1),
                               name=title, showlegend=False), row=i, col=1)
    if flag_sat:
        sx, sy = _sat(x, y)
        if sx.size:
            fig.add_trace(go.Scattergl(x=sx, y=sy, mode="markers",
                          marker=dict(color="#d62728", size=4), name="saturated",
                          showlegend=False), row=i, col=1)
fig.update_xaxes(title_text="time since session start (s)", row=len(rows), col=1)
fig.update_layout(height=760, template="plotly_white", margin=dict(t=40, b=40),
                  title=f"{sess.name} — raw channels")
fig.show(config=png_cfg(f"{sess.name}_raw"))

## Part 2 — Converted + actuation-marked

Resistance, electrical power, calibrated displacement & force, and ΔR/R₀.
Orange bands = fire windows.

In [ ]:
dRR = (R_ohm / R0 - 1.0) * 100.0   # %

panels = [
    ("Resistance R [Ω]",        rel(t_r),   R_ohm,    sp.C_RES),
    ("Power P = V·I [W]",       grid,       Pg,       sp.C_POWER),
    ("Displacement [mm]",       rel(t_las), disp_mm,  sp.C_DISP),
    ("Force [mN]",              rel(t_lod), force_mN, sp.C_FORCE),
    ("ΔR/R₀ [%]",               rel(t_r),   dRR,      sp.C_RES),
]
fig = make_subplots(rows=len(panels), cols=1, shared_xaxes=True, vertical_spacing=0.028,
                    subplot_titles=[p[0] for p in panels])
for i, (title, x, y, color) in enumerate(panels, start=1):
    fig.add_trace(go.Scattergl(x=x, y=y, mode="lines", line=dict(color=color, width=1),
                               showlegend=False), row=i, col=1)
# fire-window shading on every panel
for i in range(1, len(panels) + 1):
    for ot in onsets_rel:
        fig.add_vrect(x0=ot, x1=ot + fire_s, fillcolor=sp.C_FIRE, opacity=0.12,
                      line_width=0, row=i, col=1)
# reference lines
fig.add_hline(y=R0, line=dict(color="#898781", dash="dash", width=1), row=1, col=1)
fig.add_hline(y=0.0, line=dict(color="#898781", dash="dash", width=1), row=5, col=1)
fig.update_xaxes(title_text="time since session start (s)", row=len(panels), col=1)
fig.update_layout(height=920, template="plotly_white", margin=dict(t=40, b=40),
                  title=f"{sess.name} — converted (R0={R0:.3f} Ω)")
fig.show(config=png_cfg(f"{sess.name}_converted"))

## Part 3 — Cross-plots (parametric)

Left column shares x = **resistance [Ω]**; right column shares x = **power [W]**.
Top row shares y = **displacement [mm]**; bottom row shares y = **force [mN]**.
Points colored by phase — **fire** (heating) vs **cool** — so the hysteresis loop
is visible (heating and cooling trace different paths).

In [ ]:
COOL_C, FIRE_C = "#2a78d6", "#d62728"

fig = make_subplots(rows=2, cols=2, shared_xaxes=True, shared_yaxes=True,
                    horizontal_spacing=0.06, vertical_spacing=0.09,
                    subplot_titles=("Displacement vs R", "Displacement vs P",
                                    "Force vs R", "Force vs P"))

# (row, col): (x-array, y-array)
cells = {
    (1, 1): (Rg, dispg),  (1, 2): (Pg, dispg),
    (2, 1): (Rg, forceg), (2, 2): (Pg, forceg),
}
for (r, c), (xg, yg) in cells.items():
    first = (r == 1 and c == 1)
    for label, cmask, color in (("cool", ~fire_mask, COOL_C), ("fire", fire_mask, FIRE_C)):
        fig.add_trace(go.Scattergl(x=xg[cmask], y=yg[cmask], mode="markers",
                      marker=dict(color=color, size=3, opacity=0.45),
                      name=label, legendgroup=label, showlegend=first),
                      row=r, col=c)

fig.update_xaxes(title_text="Resistance [Ω]", row=2, col=1)
fig.update_xaxes(title_text="Power [W]",      row=2, col=2)
fig.update_yaxes(title_text="Displacement [mm]", row=1, col=1)
fig.update_yaxes(title_text="Force [mN]",        row=2, col=1)
fig.update_layout(height=760, template="plotly_white", margin=dict(t=50, b=40),
                  legend=dict(orientation="h", y=1.06, x=0.5, xanchor="center"),
                  title=f"{sess.name} — cross-plots (fire vs cool)")
fig.show(config=png_cfg(f"{sess.name}_crossplots"))

---
### Notes / how to extend

- **Save a plot:** hover a figure → camera icon (top-right) → downloads the
  current zoomed view as PNG (`scale=3`, high-DPI). Or `fig.write_html(...)`
  for an interactive standalone file.
- **Add/remove a panel:** edit the `rows` / `panels` / `cells` lists — each
  entry is one panel. No other code changes needed.
- **Different session:** set `SESSION` in the 2nd code cell and re-run.
- **Whole-session view** is used throughout (first pass). To go per-cycle /
  actuation-aligned, slice each channel on `onsets_rel` with `sp.slice_cycle`
  and overlay — say the word and I'll add a Part-2b/3b for that.
- Displacement is **raw** (unfiltered). The laser carries a ~65.8 Hz tone +
  zero-order-hold duplicates; `sp.filter_channel(...)` / `sp.notch_fft(...)` are
  imported and ready if you want an optional de-noised trace.